In [1]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("movies-app")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0")
    .master("local[*]")
    .getOrCreate())



/Users/teobun/Desktop/BigData2026/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/25 14:26:23 WARN Utils: Your hostname, Khanhs-Laptop.local, resolves to a loopback address: 127.0.0.1; using 10.128.27.29 instead (on interface en0)
26/03/25 14:26:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/teobun/Desktop/BigData2026/venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/teobun/.ivy2.5.2/cache
The jars for the packages stored in: /Users/teobun/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apac

In [2]:
path = kagglehub.dataset_download("grouplens/movielens-latest-small")
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)
df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 1 row
+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
+-------+----------------+--------------------+
only showing top 1 row
+------+-------+-----+----------+
|userId|movieId|  tag| timestamp|
+------+-------+-----+----------+
|     2|  60756|funny|1445714994|
+------+-------+-----+----------+
only showing top 1 row


In [3]:
# First, delete topics if they exist
admin_client = AdminClient({'bootstrap.servers': 'localhost:9092'})
admin_client.delete_topics(['ratings', 'movies', 'tags'], operation_timeout=10)
topic_metadata = admin_client.list_topics(timeout=10)
print("Deleted if exists")

# Create new topics
new_topics = [
    NewTopic(topic="ratings", num_partitions=1, replication_factor=3),
    NewTopic(topic="movies", num_partitions=1, replication_factor=3),
    NewTopic(topic="tags", num_partitions=1, replication_factor=3)
]

fs = admin_client.create_topics(new_topics)

# Iterate through the returned futures and check for success
for topic, f in fs.items():
    try:
        f.result()  # The result itself is None; wait for the operation to complete
        print(f"Topic {topic} created")
    except Exception as e:
        print(f"Failed to create topic {topic}: {e}")


Deleted if exists
Topic ratings created
Topic movies created
Topic tags created


In [4]:
# Push the dataframes to kafka broker
df_ratings.selectExpr("to_json(struct(*)) AS value") \
.write \
.format("kafka") \
.option("kafka.bootstrap.servers", "localhost:9092") \
.option("topic", "ratings") \
.save()
df_movies.selectExpr("to_json(struct(*)) AS value") \
.write \
.format("kafka") \
.option("kafka.bootstrap.servers", "localhost:9092") \
.option("topic", "movies") \
.save()
df_tags.selectExpr("to_json(struct(*)) AS value") \
.write \
.format("kafka") \
.option("kafka.bootstrap.servers", "localhost:9092") \
.option("topic", "tags") \
.save()
# Check if data is in Kafka topics, however it's stored in binary format, and in "value" column
df_ratings_kafka = spark.read \
.format("kafka") \
.option("kafka.bootstrap.servers", "localhost:9092") \
.option("subscribe", "ratings") \
.load()
df_ratings_kafka = df_ratings_kafka.selectExpr("CAST(value AS STRING)")
df_ratings_kafka.show(1)

+--------------------+
|               value|
+--------------------+
|{"userId":1,"movi...|
+--------------------+
only showing top 1 row


# 1. Read from Kafka and Resolve the Data Format Error


In [ ]:
from pyspark.sql.functions import col, from_json, avg, count, desc
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, LongType

# 1a. Define Schemas to parse the JSON
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True)
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", LongType(), True)
])

# 1b. Helper function to read, cast, and parse Kafka topics
def read_kafka_topic(topic_name, schema):
    # Read raw binary data from Kafka
    raw_df = spark.read \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "10.1.1.10:30090,10.1.1.27:30091,10.1.1.203:30092") \
        .option("subscribe", topic_name) \
        .option("startingOffsets", "earliest") \
        .load()
    
    # Cast to string and parse JSON into individual columns
    parsed_df = raw_df.selectExpr("CAST(value AS STRING)") \
        .select(from_json(col("value"), schema).alias("data")) \
        .select("data.*")
        
    return parsed_df

# Load the dataframes
df_ratings_k = read_kafka_topic("ratings", ratings_schema)
df_movies_k = read_kafka_topic("movies", movies_schema)
df_tags_k = read_kafka_topic("tags", tags_schema)

print("Data successfully parsed from Kafka:")
df_ratings_k.show(5)


Data successfully parsed from Kafka:


Py4JJavaError: An error occurred while calling o104.showString.
: java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.TimeoutException: Timed out waiting for a node assignment. Call: listNodes
	at java.base/java.util.concurrent.CompletableFuture.reportGet(CompletableFuture.java:396)
	at java.base/java.util.concurrent.CompletableFuture.get(CompletableFuture.java:2073)
	at org.apache.kafka.common.internals.KafkaFutureImpl.get(KafkaFutureImpl.java:165)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.retrieveAllPartitions(ConsumerStrategy.scala:65)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.retrieveAllPartitions$(ConsumerStrategy.scala:64)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.retrieveAllPartitions(ConsumerStrategy.scala:102)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.assignedTopicPartitions(ConsumerStrategy.scala:113)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.fetchPartitionOffsets(KafkaOffsetReaderAdmin.scala:130)
	at or

# 2. Get top 5 movies by average rating (count > 30)


In [ ]:
# Group by movieId, calculate average and count
movie_stats = df_ratings_k.groupBy("movieId").agg(
    avg("rating").alias("avg_rating"),
    count("rating").alias("rating_count")
)

# Filter for count > 30, sort by highest rating, and take top 5
top_5_movies = movie_stats.filter(col("rating_count") > 30) \
    .orderBy(desc("avg_rating")) \
    .limit(5)

# Join with movies dataframe to get the titles
top_5_with_titles = top_5_movies.join(df_movies_k, "movieId") \
    .select("movieId", "title", "avg_rating", "rating_count") \
    .orderBy(desc("avg_rating"))

print("Top 5 Movies (>30 ratings):")
top_5_with_titles.show(truncate=False)


# 3. Get 5 worst tags (lowest average rating)


In [ ]:
# Get unique movie-tag combinations to prevent duplication during join

unique_movie_tags = df_tags_k.select("movieId", "tag").distinct()

# Join with ratings to calculate the tag's overall average rating

tag_performance = unique_movie_tags.join(df_ratings_k, "movieId") \
 .groupBy("tag") \
 .agg(avg("rating").alias("tag_avg_rating")) \
 .orderBy("tag_avg_rating")

# Retrieve the worst 5 tags

worst_5_tags = tag_performance.limit(5)

print("5 Worst Tags:")
worst_5_tags.show()


# 4. Analyze: Do certain tags trigger lower ratings?


In [6]:
# Extract the 5 worst tags as a Python list
worst_tags_list = [row['tag'] for row in worst_5_tags.collect()]

# Filter tags to only include our worst 5, and get their movies
movies_with_worst_tags = unique_movie_tags.filter(col("tag").isin(worst_tags_list))

# 4a. Check their average ratings
analysis_df = movies_with_worst_tags.join(movie_stats, "movieId") \
    .join(df_movies_k, "movieId") \
    .select("tag", "title", "avg_rating", "rating_count") \
    .orderBy("tag", "avg_rating")

print("Average ratings for movies associated with the worst 5 tags:")
analysis_df.show(truncate=False)

# 4b. How many movies have these tags?
tag_distribution = movies_with_worst_tags.groupBy("tag").agg(
    count("movieId").alias("num_movies_with_tag")
)

print("Number of movies carrying each of the worst tags:")
tag_distribution.show()


NameError: name 'worst_5_tags' is not defined